# IMS + OL/DA Colorado QC (3x2 per day)

For each selected date, this notebook creates a **3x2** figure:

- Row 1: IMS raw (left), IMS regridded-to-M36 (right)
- Row 2: Model thresholded snow/no-snow (OL left, DA right)
- Row 3: Model snow cover fraction (OL left, DA right)

The map features are loaded explicitly from local Natural Earth shapefiles to avoid Cartopy downloader issues.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy
import cartopy.crs as ccrs
from cartopy.io import shapereader
from cartopy.feature import ShapelyFeature

YEAR = 2020

DATES_TO_PLOT = [
    pd.Timestamp("2020-01-01"),
    pd.Timestamp("2020-04-01"),
    pd.Timestamp("2020-11-01"),
]

# Colorado extent: (west, east, south, north).
CO_EXTENT = (-109.1, -102.0, 36.8, 41.2)

# IMS inputs.
IMS_RAW_FILE = Path("/gpfsm/dnb06/projects/p163/IMS/ims_snowcover_24km_2020.nc4")
IMS_REGRID_FILE = Path("/discover/nobackup/projects/land_da/geosldas-analysis/projects/IMS/output/ims_snowcover_24km_2020_on_m36_nearest.nc4")
RAW_VAR_CANDIDATES = ("ims_snowcover", "ims_category")
REGRID_VAR_CANDIDATES = ("ims_category", "ims_snowcover")
IMS_FILL_VALUES = {-32768}

# Model inputs (matching SNOTEL/IMS analysis setup).
DOMAIN = "SMAP_EASEv2_M36_GLOBAL"
EXPERIMENTS = {
    "OL": {
        "exp_name": "LS_OLv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_OLv8_M36_v2/LS_OLv8_M36"),
    },
    "DA": {
        "exp_name": "LS_DAv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_DAv8_M36_v3/LS_DAv8_M36"),
    },
}
MODEL_SCF_VAR_CANDIDATES = ("FRLANDSNO", "FRSNO", "SNCOVFR", "SNOWCOVERFR", "SCF")

# Threshold for middle-row binary snow/no-snow panels.
MODEL_BINARY_THRESHOLD = 0.0

SAVE_FIGURES = True
FIG_OUTPUT_DIR = Path("./outputs_ims_regrid_qc_colorado")

# Local Natural Earth shapefiles to avoid downloader issues.
CARTOPY_DATA_DIR_CANDIDATES = [
    Path("/home/amfox/.local/share/cartopy"),
    Path.home() / ".local" / "share" / "cartopy",
]
CARTOPY_DATA_DIR = next((p for p in CARTOPY_DATA_DIR_CANDIDATES if p.exists()), CARTOPY_DATA_DIR_CANDIDATES[0])
cartopy.config["data_dir"] = str(CARTOPY_DATA_DIR)
NATURAL_EARTH_ROOT = CARTOPY_DATA_DIR / "shapefiles" / "natural_earth"

LOCAL_COAST_SHP = NATURAL_EARTH_ROOT / "physical" / "ne_50m_coastline.shp"
LOCAL_LAND_SHP = NATURAL_EARTH_ROOT / "physical" / "ne_50m_land.shp"
LOCAL_BORDERS_SHP = NATURAL_EARTH_ROOT / "cultural" / "ne_50m_admin_0_boundary_lines_land.shp"
LOCAL_STATE_SHP_CANDIDATES = [
    NATURAL_EARTH_ROOT / "cultural" / "ne_50m_admin_1_states_provinces_lines.shp",
    NATURAL_EARTH_ROOT / "cultural" / "ne_50m_admin_1_states_provinces_lakes.shp",
]
LOCAL_STATES_SHP = next((p for p in LOCAL_STATE_SHP_CANDIDATES if p.exists()), None)

# Repo-local tilecoord reader.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "common/python/io/read_GEOSldas.py").exists():
            REPO_ROOT = parent
            break
if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    raise FileNotFoundError("Could not find common/python/io/read_GEOSldas.py from current working directory")
sys.path.insert(0, str(REPO_ROOT / "common/python/io"))
from read_GEOSldas import read_tilecoord  # type: ignore

print(f"IMS_RAW_FILE={IMS_RAW_FILE}")
print(f"IMS_REGRID_FILE={IMS_REGRID_FILE}")
print(f"DATES_TO_PLOT={[d.strftime('%Y-%m-%d') for d in DATES_TO_PLOT]}")
print(f"MODEL_BINARY_THRESHOLD={MODEL_BINARY_THRESHOLD}")
print(f"SAVE_FIGURES={SAVE_FIGURES}")
print(f"FIG_OUTPUT_DIR={FIG_OUTPUT_DIR}")
print(f"CARTOPY_DATA_DIR={CARTOPY_DATA_DIR}")
print(f"LOCAL_COAST_SHP={LOCAL_COAST_SHP}")
print(f"LOCAL_LAND_SHP={LOCAL_LAND_SHP}")
print(f"LOCAL_BORDERS_SHP={LOCAL_BORDERS_SHP}")
print(f"LOCAL_STATES_SHP={LOCAL_STATES_SHP}")
for k, cfg in EXPERIMENTS.items():
    print(f"{k}: exp_name={cfg['exp_name']}, run_root={cfg['run_root']}")


In [ ]:
def choose_var(ds: xr.Dataset, candidates):
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"None of the candidate variables were found: {candidates}")


def wrap_lon_180(lon):
    arr = np.asarray(lon, dtype=np.float64)
    return ((arr + 180.0) % 360.0) - 180.0


def decode_ims_dates(ds: xr.Dataset, var_name: str, year: int) -> pd.DatetimeIndex:
    da = ds[var_name]
    if da.ndim != 3:
        raise ValueError(f"{var_name} must be 3D; got dims={da.dims}")

    n_time = int(da.shape[0])
    time_dim = str(da.dims[0])

    if "doy" in ds.variables and ds["doy"].ndim == 1 and ds["doy"].shape[0] == n_time:
        doy = np.asarray(ds["doy"].values, dtype=float)
        base = pd.Timestamp(f"{year}-01-01")
        return pd.DatetimeIndex(base + pd.to_timedelta(doy - 1.0, unit="D"))

    for name in (time_dim, "time", "day_of_year"):
        if name not in ds.variables:
            continue
        tvar = ds[name]
        if tvar.ndim != 1 or tvar.shape[0] != n_time:
            continue

        vals = np.asarray(tvar.values)
        if np.issubdtype(vals.dtype, np.datetime64):
            return pd.DatetimeIndex(pd.to_datetime(vals))

        units = str(tvar.attrs.get("units", ""))
        if "since" in units:
            try:
                base_txt = units.split("since", 1)[1].strip().split()[0]
                base = pd.Timestamp(base_txt)
                return pd.DatetimeIndex(base + pd.to_timedelta(vals.astype(float), unit="D"))
            except Exception:
                pass

    base = pd.Timestamp(f"{year}-01-01")
    return pd.DatetimeIndex(base + pd.to_timedelta(np.arange(n_time), unit="D"))


def get_date_index(dates: pd.DatetimeIndex, target_day: pd.Timestamp) -> int:
    target = pd.Timestamp(target_day).normalize()
    idx = np.where(dates.normalize() == target)[0]
    if idx.size == 0:
        raise KeyError(f"Date {target.strftime('%Y-%m-%d')} not found in dataset")
    return int(idx[0])


def read_slice_as_yx(da: xr.DataArray, t_index: int, lat2d: np.ndarray):
    time_dim = str(da.dims[0])
    arr = np.asarray(da.isel({time_dim: t_index}).values, dtype=np.float32)

    if arr.shape == lat2d.shape:
        return arr
    if arr.T.shape == lat2d.shape:
        return arr.T

    raise ValueError(
        f"Slice shape {arr.shape} does not match lat/lon shape {lat2d.shape} (or transpose {arr.T.shape})"
    )


def prepare_ims_codes(arr2d: np.ndarray, fill_values: set[int]):
    arr = np.asarray(arr2d, dtype=np.float32)
    finite = np.isfinite(arr)

    out = np.full(arr.shape, np.nan, dtype=np.float32)
    out[finite] = np.rint(arr[finite]).astype(np.float32)

    for fv in fill_values:
        out[out == float(fv)] = np.nan

    return out


def _fill_nonfinite_2d(arr2d: np.ndarray) -> np.ndarray:
    out = np.asarray(arr2d, dtype=np.float64).copy()
    ny, nx = out.shape

    x = np.arange(nx, dtype=np.float64)
    for j in range(ny):
        row = out[j, :]
        good = np.isfinite(row)
        ng = int(np.sum(good))
        if ng == 0:
            continue
        if ng == 1:
            out[j, :] = float(row[good][0])
        elif ng < nx:
            out[j, :] = np.interp(x, x[good], row[good])

    y = np.arange(ny, dtype=np.float64)
    for i in range(nx):
        col = out[:, i]
        good = np.isfinite(col)
        ng = int(np.sum(good))
        if ng == 0:
            continue
        if ng == 1:
            out[:, i] = float(col[good][0])
        elif ng < ny:
            out[:, i] = np.interp(y, y[good], col[good])

    if np.any(~np.isfinite(out)):
        g = float(np.nanmean(out))
        if not np.isfinite(g):
            raise ValueError("Could not build finite coordinate grid for pcolormesh")
        out[~np.isfinite(out)] = g

    return out.astype(np.float32)


def make_finite_lonlat_for_pcolormesh(lon2d: np.ndarray, lat2d: np.ndarray):
    lon = np.asarray(lon2d, dtype=np.float32)
    lat = np.asarray(lat2d, dtype=np.float32)
    bad = ~np.isfinite(lon) | ~np.isfinite(lat)
    n_bad = int(np.sum(bad))
    if n_bad == 0:
        return wrap_lon_180(lon), lat, 0

    lon_f = wrap_lon_180(_fill_nonfinite_2d(lon))
    lat_f = _fill_nonfinite_2d(lat)
    if np.any(~np.isfinite(lon_f) | ~np.isfinite(lat_f)):
        raise ValueError("Coordinate fill failed; lon/lat still contain non-finite values")
    return lon_f, lat_f, n_bad


def _load_local_feature(shp_path: Path, *, facecolor="none", edgecolor="black", linewidth=0.5):
    if shp_path is None:
        return None
    shp_path = Path(shp_path)
    if not shp_path.exists():
        return None
    geoms = list(shapereader.Reader(str(shp_path)).geometries())
    if len(geoms) == 0:
        return None
    return ShapelyFeature(
        geoms,
        ccrs.PlateCarree(),
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=linewidth,
    )


def build_local_map_features():
    return {
        "land": _load_local_feature(LOCAL_LAND_SHP, facecolor="none", edgecolor="0.75", linewidth=0.2),
        "coast": _load_local_feature(LOCAL_COAST_SHP, facecolor="none", edgecolor="black", linewidth=0.5),
        "borders": _load_local_feature(LOCAL_BORDERS_SHP, facecolor="none", edgecolor="0.25", linewidth=0.3),
        "states": _load_local_feature(LOCAL_STATES_SHP, facecolor="none", edgecolor="black", linewidth=0.3),
    }


def add_local_map_features(ax, features: dict):
    if features.get("land") is not None:
        ax.add_feature(features["land"], zorder=2)
    if features.get("coast") is not None:
        ax.add_feature(features["coast"], zorder=3)
    if features.get("borders") is not None:
        ax.add_feature(features["borders"], zorder=3)
    if features.get("states") is not None:
        ax.add_feature(features["states"], zorder=3)


def locate_tilecoord_file(run_root: Path, exp_name: str, domain: str):
    candidates = [
        run_root / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / exp_name / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / f"{exp_name}.ldas_tilecoord.bin",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find tilecoord file. Checked: " + ", ".join(str(x) for x in candidates))


def locate_daily_cat_file(run_root: Path, exp_name: str, domain: str, day: pd.Timestamp):
    y = f"Y{day.year:04d}"
    m = f"M{day.month:02d}"
    stamp = day.strftime("%Y%m%d")
    fname = f"{exp_name}.tavg24_1d_lnd_Nt.{stamp}_1200z.nc4"
    candidates = [
        run_root / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / exp_name / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / "cat" / "ens_avg" / y / m / fname,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def choose_representative_tile_per_cell(tc: dict):
    i_indg = np.asarray(tc["i_indg"], dtype=np.int64)
    j_indg = np.asarray(tc["j_indg"], dtype=np.int64)
    tile_id = np.asarray(tc["tile_id"], dtype=np.int64)
    frac_cell = np.asarray(tc["frac_cell"], dtype=np.float64)

    frac_sort = np.where(np.isfinite(frac_cell), frac_cell, -np.inf)
    nx = int(i_indg.max()) + 1
    ny = int(j_indg.max()) + 1
    cell_code = j_indg * nx + i_indg

    order = np.lexsort((tile_id, -frac_sort, cell_code))
    code_sorted = cell_code[order]

    first = np.empty(code_sorted.size, dtype=bool)
    first[0] = True
    first[1:] = code_sorted[1:] != code_sorted[:-1]
    rep_idx = order[first]

    return {
        "rep_idx": rep_idx.astype(np.int64),
        "rep_i": i_indg[rep_idx].astype(np.int32),
        "rep_j": j_indg[rep_idx].astype(np.int32),
        "nx": np.int32(nx),
        "ny": np.int32(ny),
    }


def read_model_scf_grid_for_rep(nc_path: Path, rep: dict, var_candidates, forced_var_name=None):
    with xr.open_dataset(nc_path, decode_times=False) as ds:
        used_var = forced_var_name if forced_var_name is not None else choose_var(ds, var_candidates)
        if used_var not in ds.variables:
            raise KeyError(f"Model SCF variable '{used_var}' not found in {nc_path}")

        da = ds[used_var]
        if "time" in da.dims:
            da = da.isel(time=0)
        vals = np.asarray(da.values, dtype=np.float32).reshape(-1)

    rep_idx = rep["rep_idx"]
    if vals.size <= int(rep_idx.max()):
        raise ValueError(
            f"Model var length {vals.size} smaller than representative index max {int(rep_idx.max())}"
        )

    sel = np.asarray(vals[rep_idx], dtype=np.float32)
    sel[(sel > 1e14) | (sel < 0.0)] = np.nan

    ny = int(rep["ny"])
    nx = int(rep["nx"])
    grid = np.full((ny, nx), np.nan, dtype=np.float32)
    grid[rep["rep_j"], rep["rep_i"]] = sel
    return grid, used_var


# Colormaps
IMS_CODES = [0, 1, 2, 3, 4]
IMS_LABELS = [
    "0: outside coverage",
    "1: water",
    "2: land no snow",
    "3: sea ice",
    "4: snow",
]
IMS_CMAP = ListedColormap(["#cfcfcf", "#2b83ba", "#c7a76c", "#98d5ef", "#ffffff"])
IMS_NORM = BoundaryNorm(np.array([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]), IMS_CMAP.N)

MODEL_BIN_CMAP = ListedColormap(["#c7a76c", "#ffffff"])
MODEL_BIN_NORM = BoundaryNorm(np.array([-0.5, 0.5, 1.5]), MODEL_BIN_CMAP.N)


In [ ]:
# Input checks and static setup.
if not IMS_RAW_FILE.exists():
    raise FileNotFoundError(f"Raw IMS file not found: {IMS_RAW_FILE}")
if not IMS_REGRID_FILE.exists():
    raise FileNotFoundError(f"Regridded IMS file not found: {IMS_REGRID_FILE}")
for p in (LOCAL_COAST_SHP, LOCAL_LAND_SHP, LOCAL_BORDERS_SHP):
    if not Path(p).exists():
        raise FileNotFoundError(f"Required local shapefile missing: {p}")
if LOCAL_STATES_SHP is None:
    print("Note: local admin-1 states shapefile not found; state boundaries will be skipped.")

local_features = build_local_map_features()

ds_raw = xr.open_dataset(IMS_RAW_FILE, decode_times=False)
ds_regrid = xr.open_dataset(IMS_REGRID_FILE, decode_times=False)

raw_var = choose_var(ds_raw, RAW_VAR_CANDIDATES)
regrid_var = choose_var(ds_regrid, REGRID_VAR_CANDIDATES)

raw_dates = decode_ims_dates(ds_raw, raw_var, YEAR)
regrid_dates = decode_ims_dates(ds_regrid, regrid_var, YEAR)

raw_lat = np.asarray(ds_raw["lat"].values, dtype=np.float32)
raw_lon = wrap_lon_180(np.asarray(ds_raw["lon"].values, dtype=np.float32))

re_lat = np.asarray(ds_regrid["lat"].values, dtype=np.float32)
re_lon = wrap_lon_180(np.asarray(ds_regrid["lon"].values, dtype=np.float32))

tilecoord_path = locate_tilecoord_file(
    run_root=Path(EXPERIMENTS["OL"]["run_root"]),
    exp_name=str(EXPERIMENTS["OL"]["exp_name"]),
    domain=DOMAIN,
)
tc = read_tilecoord(str(tilecoord_path))
rep = choose_representative_tile_per_cell(tc)

if (int(rep["ny"]), int(rep["nx"])) != re_lat.shape:
    raise ValueError(
        f"Representative grid shape {(int(rep['ny']), int(rep['nx']))} does not match regridded IMS shape {re_lat.shape}"
    )

print(f"Raw var={raw_var}, shape={ds_raw[raw_var].shape}")
print(f"Regridded var={regrid_var}, shape={ds_regrid[regrid_var].shape}")
print(f"Raw date span: {raw_dates.min()} to {raw_dates.max()}")
print(f"Regrid date span: {regrid_dates.min()} to {regrid_dates.max()}")
print(f"Tilecoord: {tilecoord_path}")
print(f"Representative cells: {rep['rep_idx'].size}")


In [ ]:
exp_model_var = {"OL": None, "DA": None}
if SAVE_FIGURES:
    FIG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


for day in DATES_TO_PLOT:
    i_raw = get_date_index(raw_dates, day)
    i_re = get_date_index(regrid_dates, day)

    raw_slice = read_slice_as_yx(ds_raw[raw_var], i_raw, raw_lat)
    re_slice = read_slice_as_yx(ds_regrid[regrid_var], i_re, re_lat)

    raw_plot = prepare_ims_codes(raw_slice, IMS_FILL_VALUES)
    re_plot = prepare_ims_codes(re_slice, IMS_FILL_VALUES)

    model_scf = {}
    model_bin = {}
    model_meta = {}

    for exp_key in ("OL", "DA"):
        cfg = EXPERIMENTS[exp_key]
        run_root = Path(cfg["run_root"])
        exp_name = str(cfg["exp_name"])

        model_path = locate_daily_cat_file(run_root, exp_name, DOMAIN, day)
        if model_path is None:
            raise FileNotFoundError(f"Model file not found for {exp_key} on {day.strftime('%Y-%m-%d')}")

        grid, used_var = read_model_scf_grid_for_rep(
            model_path,
            rep=rep,
            var_candidates=MODEL_SCF_VAR_CANDIDATES,
            forced_var_name=exp_model_var.get(exp_key),
        )
        if exp_model_var.get(exp_key) is None:
            exp_model_var[exp_key] = used_var

        model_scf[exp_key] = grid
        b = np.full(grid.shape, np.nan, dtype=np.float32)
        valid = np.isfinite(grid)
        b[valid] = (grid[valid] > np.float32(MODEL_BINARY_THRESHOLD)).astype(np.float32)
        model_bin[exp_key] = b
        model_meta[exp_key] = (used_var, model_path)

    fig, axes = plt.subplots(
        3,
        2,
        figsize=(13, 14),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )

    panel_specs = [
        (0, 0, raw_lon, raw_lat, raw_plot, "ims", "IMS 24km raw"),
        (0, 1, re_lon, re_lat, re_plot, "ims", "IMS regridded to M36 (nearest)"),
        (1, 0, re_lon, re_lat, model_bin["OL"], "bin", f"OL binary snow (SCF > {MODEL_BINARY_THRESHOLD:g})"),
        (1, 1, re_lon, re_lat, model_bin["DA"], "bin", f"DA binary snow (SCF > {MODEL_BINARY_THRESHOLD:g})"),
        (2, 0, re_lon, re_lat, model_scf["OL"], "scf", f"OL SCF ({model_meta['OL'][0]})"),
        (2, 1, re_lon, re_lat, model_scf["DA"], "scf", f"DA SCF ({model_meta['DA'][0]})"),
    ]

    row_mappable = {0: None, 1: None, 2: None}

    for r, c, lon2d, lat2d, arr2d, kind, title in panel_specs:
        ax = axes[r, c]
        lon_plot, lat_plot, n_filled = make_finite_lonlat_for_pcolormesh(lon2d, lat2d)

        if kind == "ims":
            m = ax.pcolormesh(
                lon_plot, lat_plot, arr2d, cmap=IMS_CMAP, norm=IMS_NORM,
                transform=ccrs.PlateCarree(), shading="auto"
            )
        elif kind == "bin":
            m = ax.pcolormesh(
                lon_plot, lat_plot, arr2d, cmap=MODEL_BIN_CMAP, norm=MODEL_BIN_NORM,
                transform=ccrs.PlateCarree(), shading="auto"
            )
        else:
            m = ax.pcolormesh(
                lon_plot, lat_plot, arr2d, cmap="Blues", vmin=0.0, vmax=1.0,
                transform=ccrs.PlateCarree(), shading="auto"
            )

        if row_mappable[r] is None:
            row_mappable[r] = m

        ax.set_extent(CO_EXTENT, crs=ccrs.PlateCarree())
        add_local_map_features(ax, local_features)

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if c == 1:
            gl.left_labels = False
        if r < 2:
            gl.bottom_labels = False

        if n_filled > 0:
            ax.set_title(f"{title} (filled {n_filled} non-finite lon/lat)")
        else:
            ax.set_title(title)

    cb0 = fig.colorbar(row_mappable[0], ax=axes[0, :], shrink=0.9, pad=0.02)
    cb0.set_ticks(IMS_CODES)
    cb0.set_ticklabels(IMS_LABELS)

    cb1 = fig.colorbar(row_mappable[1], ax=axes[1, :], shrink=0.9, pad=0.02)
    cb1.set_ticks([0, 1])
    cb1.set_ticklabels(["no snow", "snow"])

    cb2 = fig.colorbar(row_mappable[2], ax=axes[2, :], shrink=0.9, pad=0.02)
    cb2.set_label("snow cover fraction")

    fig.suptitle(f"Colorado IMS + OL/DA snow cover QC: {day.strftime('%Y-%m-%d')}", fontsize=14)
    if SAVE_FIGURES:
        out_png = FIG_OUTPUT_DIR / f"ims_ol_da_qc_colorado_{day.strftime('%Y%m%d')}.png"
        fig.savefig(out_png, dpi=180, bbox_inches="tight")
        print(f"Saved figure: {out_png}")
    plt.show()


In [ ]:
# Optional quick numeric check for IMS categories over Colorado.
def code_counts_in_extent(data2d, lon2d, lat2d, extent, valid_codes=(0, 1, 2, 3, 4)):
    w, e, s, n = extent
    m = (lon2d >= w) & (lon2d <= e) & (lat2d >= s) & (lat2d <= n) & np.isfinite(data2d)
    if not np.any(m):
        return {int(c): 0 for c in valid_codes}
    vals = np.rint(data2d[m]).astype(np.int32)
    return {int(c): int(np.sum(vals == int(c))) for c in valid_codes}

for day in DATES_TO_PLOT:
    i_raw = get_date_index(raw_dates, day)
    i_re = get_date_index(regrid_dates, day)

    raw_slice = prepare_ims_codes(read_slice_as_yx(ds_raw[raw_var], i_raw, raw_lat), IMS_FILL_VALUES)
    re_slice = prepare_ims_codes(read_slice_as_yx(ds_regrid[regrid_var], i_re, re_lat), IMS_FILL_VALUES)

    c_raw = code_counts_in_extent(raw_slice, raw_lon, raw_lat, CO_EXTENT)
    c_re = code_counts_in_extent(re_slice, re_lon, re_lat, CO_EXTENT)

    print(day.strftime('%Y-%m-%d'))
    print('  raw IMS   :', c_raw)
    print('  regrid IMS:', c_re)

# Close datasets when done.
# ds_raw.close(); ds_regrid.close()
